# Omni-AD-30 全量运行 —— Swin 主干版
Swin Transformer（`swin_t`，28×28+14×14 多尺度特征）+ PatchCore。30 类全量训练 / 预测 / 评测。
> 运行前：右上角「代码执行程序 → 更改运行时类型 → T4 GPU」；数据 zip 已在 Drive `MyDrive/IAD/`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
%%bash
mkdir -p /content/data
# -oq 静默覆盖，重跑安全；解压约 3~5 分钟
unzip -oq /content/drive/MyDrive/IAD/Omni-AD-30-release.zip -d /content/data/
ls /content/data/Omni-AD-30-release | wc -l   # 期望 30

In [ ]:
%%bash
cd /content
rm -rf IAD-Industrial-Anomaly-Detection
git clone -b swin https://github.com/coder-yu-WICK/IAD-Industrial-Anomaly-Detection.git
cd IAD-Industrial-Anomaly-Detection
git log --oneline -2
pip install -q onnx onnxscript onnxruntime-gpu scikit-learn

In [ ]:
%%bash
cd /content/IAD-Industrial-Anomaly-Detection
mkdir -p data
ln -s /content/data/Omni-AD-30-release data/Omni-AD-30-release
python -u src/data/sample_manifest.py --data-root data/Omni-AD-30-release

In [ ]:
%%bash
cd /content/IAD-Industrial-Anomaly-Detection
python -u src/train.py \
  --data-root data/Omni-AD-30-release \
  --manifest data/Omni-AD-30-release/train_manifest.csv \
  --output-dir work/model_swin \
  --device cuda:0 --seed 2026 --num-workers 4
# 首次自动下载 swin_t 权重(~110MB)；30 类约 30~90 分钟

In [ ]:
%%bash
cd /content/IAD-Industrial-Anomaly-Detection
python -u src/predict.py \
  --data-root data/Omni-AD-30-release \
  --manifest data/Omni-AD-30-release/test_manifest.csv \
  --model-dir work/model_swin \
  --output-dir work/pred_swin \
  --device cuda:0 --num-workers 4

In [ ]:
%%bash
cd /content/IAD-Industrial-Anomaly-Detection
python src/evaluate.py \
  --predictions-dir work/pred_swin \
  --data-root data/Omni-AD-30-release \
  --manifest data/Omni-AD-30-release/test_manifest.csv